# SØK1005 Assignment 4: Hypothesis testing, confidence intervals and p-values

****** WE chose to do the whole lab in jupyterNotebook/Python. Claude code has been used to structure the lab and to consolidate. *****

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import wooldridge

pd.set_option("display.float_format", "{:.4f}".format)

### A helper for t-tests

`t_test()` runs a t-test on a single coefficient and prints each step. The test statistic is

$$t = \frac{\hat\beta - b}{se(\hat\beta)}$$

where $b$ is the value under H0. `sided` is `"two"`, `"greater"` or `"less"` depending on H1.
Critical values and p-values come from `scipy.stats.t`.

In [2]:
def t_test(beta, se, h0, dof, alpha, sided="two", name="β"):
    t = (beta - h0) / se
    if sided == "two":
        h1, crit = f"{name} ≠ {h0}", stats.t.ppf(1 - alpha / 2, dof)
        p, reject = 2 * stats.t.sf(abs(t), dof), abs(t) > crit
        rule = f"|t| > {crit:.3f}"
    elif sided == "greater":
        h1, crit = f"{name} > {h0}", stats.t.ppf(1 - alpha, dof)
        p, reject = stats.t.sf(t, dof), t > crit
        rule = f"t > {crit:.3f}"
    elif sided == "less":
        h1, crit = f"{name} < {h0}", -stats.t.ppf(1 - alpha, dof)
        p, reject = stats.t.cdf(t, dof), t < crit
        rule = f"t < {crit:.3f}"
    else:
        raise ValueError("sided must be 'two', 'greater' or 'less'")
    h0_sign = {"two": "=", "greater": "≤", "less": "≥"}[sided]

    print(f"1. Hypotheses:         H0: {name} {h0_sign} {h0}   vs   H1: {h1}")
    print(f"2. Significance level: α = {alpha:.0%}")
    print(f"3. Test statistic:     t = (β̂ - {h0}) / se = ({beta} - {h0}) / {se} = {t:.3f}")
    print(f"4. Degrees of freedom: {dof:.0f}, critical value: reject H0 if {rule}")
    print(f"5. p-value:            p = {p:.4f}")
    print(f"6. Conclusion:         {'Reject' if reject else 'Do not reject'} H0 at the {alpha:.0%} level"
          f" ({'p < α' if reject else 'p ≥ α'}).")
    return {"t": t, "crit": crit, "p": p, "reject": reject}

# Part 1: The demand for children

Table 1 as a DataFrame, with t-values and two-sided p-values. Degrees of freedom: 72 − 10 = 62.

In [3]:
variables = ["PersonalExemption", "MaleAndAssetIncome", "Unemployment", "InfantMortality",
             "Immigration", "FemaleWage", "Pill", "WW2", "TimeTrend", "Constant"]
coef = [0.178, 0.0035, -68.12, 0.393, 964.13, 15.427, -25.383, -29.419, -0.843, 55.944]
se   = [0.0977, 0.0031, 25.818, 0.321, 329.44, 5.286, 11.961, 8.057, 0.543, 25.831]

N, k = 72, 10
dof = N - k   # degrees of freedom

table1 = pd.DataFrame({"coef": coef, "se": se}, index=variables)
table1["t"] = table1["coef"] / table1["se"]
table1["p"] = 2 * stats.t.sf(table1["t"].abs(), dof)
table1

,coef,se,t,p
PersonalExemption,0.1780,0.0977,1.8219,0.0733
MaleAndAssetIncome,0.0035,0.0031,1.1290,0.2632
Unemployment,-68.1200,25.8180,-2.6385,0.0105
InfantMortality,0.3930,0.3210,1.2243,0.2255
Immigration,964.1300,329.4400,2.9266,0.0048
FemaleWage,15.4270,5.2860,2.9185,0.0049
Pill,-25.3830,11.9610,-2.1221,0.0378
WW2,-29.4190,8.0570,-3.6514,0.0005
TimeTrend,-0.8430,0.5430,-1.5525,0.1256
Constant,55.9440,25.8310,2.1658,0.0342


## 1a) Interpretation

Unemployment and Immigration are shares between 0 and 1, so a 1 percentage point increase is +0.01.
The effect per percentage point is the coefficient divided by 100:

In [4]:
effect_per_pp = table1.loc[["Unemployment", "Immigration"], "coef"] / 100
effect_per_pp

Unemployment   -0.6812
Immigration     9.6413
Name: coef, dtype: float64

All interpretations are *all else equal*. This is time series data, not an experiment, so I read
the coefficients as associations rather than causal effects (see 1f). Significance is based on the
two-sided p-values in the table.

- **PersonalExemption (0.178):** One extra dollar of exemption goes with 0.178 more births per 1000
  women, so 100 dollars gives about 17.8. Significant at 10 % (p = 0.073), and at 5 % with the
  one-sided test in 1b.
- **MaleAndAssetIncome (0.0035):** 1000 dollars more in family income (not counting the woman's
  earnings) goes with 3.5 more births per 1000 women. Not significant (p = 0.26).
- **Unemployment (−68.12):** One percentage point higher unemployment goes with 0.68 fewer births per
  1000 women. People may put off having children when jobs are uncertain. Significant at 5 % (p = 0.011).
- **InfantMortality (0.393):** One more infant death per 1000 live births goes with 0.39 more births,
  which could reflect families replacing children who died. Not significant (p = 0.23).
- **Immigration (964.13):** One percentage point more foreign-born people goes with about 9.6 more
  births per 1000 women. The coefficient only looks huge because the variable is a share. Significant
  at 1 % (p = 0.005).
- **FemaleWage (15.427):** One dollar higher after-tax wage for women goes with 15.4 more births per
  1000 women. I'd expect a negative sign, since higher wages make it more expensive to stay home with
  children. The positive sign suggests the income effect dominates, or that the variable picks up
  general prosperity. Significant at 1 % (p = 0.005).
- **Pill (−25.383):** Fertility was 25.4 births per 1000 women lower in 1963 to 1984 than in earlier
  years. The dummy also captures anything else that changed in those years. Significant at 5 %
  (p = 0.038).
- **WW2 (−29.419):** 29.4 fewer births per 1000 women in the war years than in other years.
  Significant at 1 % (p = 0.0005).
- **TimeTrend (−0.843):** Fertility falls by 0.84 births per 1000 women each year beyond what the
  other variables explain, about 60 over the whole period. Not significant (p = 0.13).
- **Constant (55.944):** Predicted fertility when every variable is 0: no exemption, no income, no
  wages, no unemployment, no infant deaths, no immigrants and TimeTrend = 0 (the year 1912). No such
  year exists in the data, so the constant has no useful meaning on its own.

**R² = 0.829:** The model explains 83 % of the variation in fertility. That's high, but time series
that trend often give a high R² even when the relationship is spurious (see 1f), and R² says nothing
about causality.

## 1b) Does a higher exemption increase the demand for children? (5 %)

Theory says lower costs should mean more children, so I use a one-sided test:
$H_0: \beta_1 \le 0$ against $H_1: \beta_1 > 0$.

In [5]:
b1 = table1.loc["PersonalExemption", "coef"]
se1 = table1.loc["PersonalExemption", "se"]

test_1b = t_test(b1, se1, h0=0, dof=dof, alpha=0.05, sided="greater", name="β1")
print()
test_1b_two_sided = t_test(b1, se1, h0=0, dof=dof, alpha=0.05, sided="two", name="β1")

1. Hypotheses:         H0: β1 ≤ 0   vs   H1: β1 > 0
2. Significance level: α = 5%
3. Test statistic:     t = (β̂ - 0) / se = (0.178 - 0) / 0.0977 = 1.822
4. Degrees of freedom: 62, critical value: reject H0 if t > 1.670
5. p-value:            p = 0.0366
6. Conclusion:         Reject H0 at the 5% level (p < α).

1. Hypotheses:         H0: β1 = 0   vs   H1: β1 ≠ 0
2. Significance level: α = 5%
3. Test statistic:     t = (β̂ - 0) / se = (0.178 - 0) / 0.0977 = 1.822
4. Degrees of freedom: 62, critical value: reject H0 if |t| > 1.999
5. p-value:            p = 0.0733
6. Conclusion:         Do not reject H0 at the 5% level (p ≥ α).


t = 1.82 is above the one-sided critical value of 1.670 (p = 0.037), so I reject H0. The data
supports the claim that a higher exemption goes with higher fertility.

With a two-sided test the critical value is 1.999 and p = 0.073, so H0 would not be rejected at 5 %.
The result depends on ruling out a negative effect in advance, which makes it weaker than it looks.

## 1c) The pill and fertility (10 %, two-sided)

In [6]:
test_1c = t_test(table1.loc["Pill", "coef"], table1.loc["Pill", "se"],
                 h0=0, dof=dof, alpha=0.10, sided="two", name="β7")

1. Hypotheses:         H0: β7 = 0   vs   H1: β7 ≠ 0
2. Significance level: α = 10%
3. Test statistic:     t = (β̂ - 0) / se = (-25.383 - 0) / 11.961 = -2.122
4. Degrees of freedom: 62, critical value: reject H0 if |t| > 1.670
5. p-value:            p = 0.0378
6. Conclusion:         Reject H0 at the 10% level (p < α).


$H_0: \beta_7 = 0$ against $H_1: \beta_7 \ne 0$. |t| = 2.12 is above 1.670 and p = 0.038 < 0.10, so I
reject H0. The pill period is significantly associated with lower fertility (also at 5 %, but not at
1 %). We need to be careful, though: the dummy covers everything that changed after 1963, like more
women getting an education, so we can't say the pill alone caused the drop.

## 1d) 95 % confidence interval for WW2

$$\hat\beta_8 \pm t_{0.025,\,62}\cdot se(\hat\beta_8)$$

In [7]:
t_crit = stats.t.ppf(0.975, dof)
table1["lower"] = table1["coef"] - t_crit * table1["se"]
table1["upper"] = table1["coef"] + t_crit * table1["se"]
table1.loc["WW2", ["coef", "se", "lower", "upper"]]

coef    -29.4190
se        8.0570
lower   -45.5247
upper   -13.3133
Name: WW2, dtype: float64

The interval is −29.42 ± 1.999 · 8.057 = [−45.5, −13.3] births per 1000 women.

If we repeated the study many times and built an interval the same way each time, 95 % of those
intervals would contain the true β8. That doesn't mean there's a 95 % probability that β8 is in this
particular interval. β8 is a fixed number, and this interval either contains it or it doesn't.
Since 0 is outside the interval, WW2 is significant at 5 %, and even the smallest effect in the
interval (−13) is large.

## 1e) Is the effect of MaleAndAssetIncome equal to 1? (5 %, two-sided)

In [8]:
test_1e = t_test(table1.loc["MaleAndAssetIncome", "coef"], table1.loc["MaleAndAssetIncome", "se"],
                 h0=1, dof=dof, alpha=0.05, sided="two", name="β2")

1. Hypotheses:         H0: β2 = 1   vs   H1: β2 ≠ 1
2. Significance level: α = 5%
3. Test statistic:     t = (β̂ - 1) / se = (0.0035 - 1) / 0.0031 = -321.452
4. Degrees of freedom: 62, critical value: reject H0 if |t| > 1.999
5. p-value:            p = 0.0000
6. Conclusion:         Reject H0 at the 5% level (p < α).


t = (0.0035 − 1) / 0.0031 ≈ −321.5, far beyond ±1.999, and p ≈ 0. I reject H0.

The t-value is extreme because the estimate is more than 300 standard errors away from 1. But 1 isn't
a meaningful hypothesis here. The unit is births per 1000 women per dollar, so β2 = 1 would mean that
1000 extra dollars gives one more child per woman. The test is technically correct but tells us
nothing useful. β2 = 0 is the natural hypothesis, and that one can't be rejected (p = 0.26).

## 1f) Critical assessment

In [9]:
# Effect of a 100 dollar higher exemption, with 95 % CI
table1.loc["PersonalExemption", ["coef", "lower", "upper"]] * 100

coef    17.8000
lower   -1.7300
upper   37.3300
Name: PersonalExemption, dtype: float64

**Time series problems.** Fertility in one year is probably correlated with fertility the year
before (autocorrelation). Then the usual standard errors are too small and the t-values too large, so
the borderline result for PersonalExemption (t = 1.82) may not hold up. Most of the variables also
trend over time, which can produce a high R² and significant coefficients without any real
relationship (spurious regression).

**Endogeneity.** Omitted variables like women's education, house prices, childcare and abortion laws
probably affect fertility and are correlated with the regressors. There may also be reverse
causality: politicians could raise the exemption *because* birth rates are falling.

**External validity.** The US from 1913 to 1984 includes two world wars, the Great Depression and
the baby boom. Today, low fertility has more to do with people having children later, longer
education and housing costs, so the results may not carry over.

**Size versus cost.** 100 dollars more goes with about 17.8 more births per 1000 women, but the 95 %
interval is [−1.7, 37.3], which includes 0. The exemption also goes to every family that would have
had children anyway, so the cost per extra child is high. And 100 dollars back then was worth far
more than today.

**Conclusion:** I would not recommend tax exemptions as the main tool against falling fertility based
on this study. The effect is only borderline significant, the standard errors are probably too small,
and the causal interpretation is weak. Better evidence would come from reforms where some families
got the exemption and others didn't.

# Part 2: Parents' education and birth weight

In [10]:
data = wooldridge.data("bwght2")
wooldridge.data("bwght2", description=True)

name of dataset: bwght2
no of variables: 23
no of observations: 1832

+----------+---------------------------------+
| variable | label                           |
+----------+---------------------------------+
| mage     | mother's age, years             |
| meduc    | mother's educ, years            |
| monpre   | month prenatal care began       |
| npvis    | total number of prenatal visits |
| fage     | father's age, years             |
| feduc    | father's educ, years            |
| bwght    | birth weight, grams             |
| omaps    | one minute apgar score          |
| fmaps    | five minute apgar score         |
| cigs     | avg cigarettes per day          |
| drink    | avg drinks per week             |
| lbw      | =1 if bwght <= 2000             |
| vlbw     | =1 if bwght <= 1500             |
| male     | =1 if baby male                 |
| mwhte    | =1 if mother white              |
| mblck    | =1 if mother black              |
| moth     | =1 if mother is other   

In [11]:
model_vars = ["bwght", "cigs", "drink", "meduc", "feduc", "mage"]
print(data[model_vars].isna().sum())

data_clean = data.dropna(subset=model_vars)
dropped = len(data) - len(data_clean)
print(f"\nTotal: {len(data)}, used in the model: {len(data_clean)}, dropped: {dropped}")

bwght      0
cigs     110
drink    115
meduc     30
feduc     47
mage       0
dtype: int64

Total: 1832, used in the model: 1665, dropped: 167


The variables I need are `bwght` (birth weight in grams), `cigs` (cigarettes per day), `drink`
(drinks per week, the name is `drink` and not `drinks`), `meduc` and `feduc` (years of education)
and `mage` (mother's age).

Some values are missing in `cigs`, `drink`, `meduc` and `feduc`, so I drop every row with a missing
value in one of the model variables. That leaves 1665 of 1832 observations (167 dropped), which is
the same sample statsmodels would use on its own. If the missing answers aren't random, for example
if smokers skip the question more often, this could bias the results a little.

## 2a) OLS estimation

In [12]:
model = smf.ols("bwght ~ cigs + drink + meduc + feduc + mage", data=data_clean).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  bwght   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     2.834
Date:                Sun, 11 Oct 2026   Prob (F-statistic):             0.0148
Time:                        19:47:30   Log-Likelihood:                -12916.
No. Observations:                1665   AIC:                         2.584e+04
Df Residuals:                    1659   BIC:                         2.588e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3210.1460    116.760     27.494      0.0

$$
\widehat{bwght} = \underset{(116.76)}{3210.15}
\;-\; \underset{(3.51)}{9.70}\,cigs
\;-\; \underset{(48.21)}{11.64}\,drink
\;-\; \underset{(8.46)}{1.80}\,meduc
\;+\; \underset{(7.64)}{9.76}\,feduc
\;+\; \underset{(3.12)}{3.34}\,mage
$$

$$N = 1665, \qquad R^2 = 0.008$$

Standard errors in parentheses. All interpretations are all else equal:

- **cigs (−9.70):** Each extra cigarette per day goes with 9.7 grams lower birth weight. Significant at
  1 % (p = 0.006).
- **drink (−11.64):** Each extra drink per week goes with 11.6 grams lower birth weight, but the
  standard error is 48, so the estimate is very imprecise (p = 0.81). Almost no mothers in the sample
  report drinking, so there's little variation to learn from.
- **meduc (−1.80):** One more year of the mother's education goes with 1.8 grams lower birth weight.
  That's basically zero, and not significant (p = 0.83).
- **feduc (9.76):** One more year of the father's education goes with 9.8 grams higher birth weight.
  Not significant (p = 0.20). The parents' education levels are strongly correlated, which makes it
  hard to separate the two.
- **mage (3.34):** A mother who is one year older goes with 3.3 grams higher birth weight. Not
  significant (p = 0.29).
- **Constant (3210):** The predicted weight for a 0-year-old mother with no education, which has no
  real meaning.

**R² = 0.008:** The model explains less than 1 % of the variation in birth weight. Most of it comes
from things that aren't in the model, like genetics, length of pregnancy and the baby's sex. A low R²
doesn't mean the coefficients are biased, though.

In [13]:
mean_weight = data_clean["bwght"].mean()
sd_weight = data_clean["bwght"].std()
effect_10_cigs = model.params["cigs"] * 10

print(f"Mean birth weight: {mean_weight:.0f} g, standard deviation: {sd_weight:.0f} g")
print(f"10 cigarettes a day: {effect_10_cigs:.1f} g "
      f"({effect_10_cigs / mean_weight:.1%} of the mean, {effect_10_cigs / sd_weight:.2f} standard deviations)")

Mean birth weight: 3409 g, standard deviation: 568 g
10 cigarettes a day: -97.0 g (-2.8% of the mean, -0.17 standard deviations)


Ten cigarettes a day goes with about 97 grams lower birth weight. That's around 3 % of the average
weight (3409 g), or 0.17 standard deviations. For a typical baby that's not much, but it matters for
babies who are already close to the low birth weight limit of 2500 g, and across a whole population
it's a real shift. The education effects are small in comparison: four more years of the father's
education goes with about 39 grams, and that isn't significant. These are associations, not proven
effects. Smokers may also differ in other ways that affect birth weight.

## 2b) Is cigs significant at the 1 % level? (two-sided)

In [14]:
test_2b = t_test(beta=round(model.params["cigs"], 4), se=round(model.bse["cigs"], 4),
                 h0=0, dof=model.df_resid, alpha=0.01, sided="two", name="β_cigs")

1. Hypotheses:         H0: β_cigs = 0   vs   H1: β_cigs ≠ 0
2. Significance level: α = 1%
3. Test statistic:     t = (β̂ - 0) / se = (-9.7045 - 0) / 3.5102 = -2.765
4. Degrees of freedom: 1659, critical value: reject H0 if |t| > 2.579
5. p-value:            p = 0.0058
6. Conclusion:         Reject H0 at the 1% level (p < α).


$H_0: \beta_{cigs} = 0$ against $H_1: \beta_{cigs} \ne 0$. t = −9.70 / 3.51 = −2.76 with
1665 − 6 = 1659 degrees of freedom. The critical value is 2.579. Since |t| > 2.579 and
p = 0.006 < 0.01, I reject H0. Smoking during pregnancy is significantly associated with lower birth
weight, even at the 1 % level.

## 2c) Two extra control variables

A left-out variable causes bias when it affects birth weight and is also correlated with one of the
variables in the model.

1. **`npvis`, the number of prenatal visits.** Check-ups can catch problems early, so they probably
   affect birth weight. Mothers with more education tend to go more often and smokers less often.
   Leaving it out means part of the effect of prenatal care can show up as an effect of education or
   smoking. `monpre`, the month prenatal care started, would work for the same reason.
2. **`male`, whether the baby is a boy.** Boys are heavier than girls at birth on average. The baby's
   sex is random, so it won't bias the other coefficients much, but it explains part of the variation
   and makes the estimates more precise.

`fage` (father's age) is another option, since it's closely related to the mother's age and the
father's education.

## 2d) Significance in the illustrative table

In [15]:
table_2d = pd.DataFrame(
    {"coef": [-0.42, -0.18, 0.11, 0.04],
     "se":   [0.14, 0.23, 0.05, 0.05],
     "p":    [0.003, 0.435, 0.028, 0.424]},
    index=["cigs", "drinks", "motheduc", "fatheduc"])

for alpha in [0.01, 0.05, 0.10]:
    table_2d[f"sig. {alpha:.0%}"] = table_2d["p"] < alpha

table_2d

,coef,se,p,sig. 1%,sig. 5%,sig. 10%
cigs,-0.4200,0.1400,0.0030,True,True,True
drinks,-0.1800,0.2300,0.4350,False,False,False
motheduc,0.1100,0.0500,0.0280,False,True,True
fatheduc,0.0400,0.0500,0.4240,False,False,False


| Level | Significant variables |
|---|---|
| 1 % | cigs |
| 5 % | cigs, motheduc |
| 10 % | cigs, motheduc |

`drinks` and `fatheduc` aren't significant at any of these levels.

**What a p-value means:** Assume H0 is true, so the real effect is zero. The p-value is the
probability of getting an estimate at least as far from zero as the one we got, just by chance. For
`cigs`, p = 0.003 means that would happen 0.3 % of the time. It is not the probability that H0 is
true.

**Why a high p-value doesn't mean the effect is zero:** A high p-value only means we can't reject H0.
It doesn't prove H0. The data fit an effect of zero, but they also fit many other values. A high
p-value can come from a large standard error (for `drinks`, se = 0.23 is bigger than the estimate
itself), a small sample or little variation in the variable, or multicollinearity (the parents'
education levels move together, so they're hard to separate). The 95 % interval for `drinks` is about
[−0.63, 0.27], which includes both zero and fairly large negative effects. "Not significant" means we
don't know enough, not that there is no effect.

## 2e) Confidence intervals for meduc and feduc

In [16]:
ci = model.conf_int(alpha=0.05)
ci.columns = ["lower", "upper"]
ci["contains_0"] = (ci["lower"] < 0) & (ci["upper"] > 0)
ci.loc[["meduc", "feduc"]]

,lower,upper,contains_0
meduc,-18.3886,14.7885,True
feduc,-5.2202,24.7343,True


Both intervals contain 0 (meduc: [−18.4, 14.8], feduc: [−5.2, 24.7]), so neither the mother's nor
the father's education is significant at 5 %. The intervals are also wide. For the father's
education, anything from −5 to +25 grams per year fits the data, so a meaningful positive effect
can't be ruled out.

A 95 % confidence interval contains exactly the values that would not be rejected in a two-sided test
at 5 %. So H0: β = b is rejected if and only if b lies outside the interval. Testing whether a
coefficient is zero is therefore the same as checking whether 0 is inside its interval.

# Summary

- **1a:** Unemployment (−0.68 births per percentage point), Immigration (+9.6 per percentage point),
  FemaleWage, Pill (−25.4) and WW2 (−29.4) are significant at 5 % or better. PersonalExemption is
  significant at 10 %. The constant has no real meaning, and the high R² may be partly spurious.
- **1b:** One-sided, t = 1.82 > 1.670, so I reject H0. A two-sided test would not reject (p = 0.073).
- **1c:** t = −2.12 and p = 0.038 < 0.10, so the pill period is significantly linked to lower fertility.
- **1d:** The 95 % interval for WW2 is [−45.5, −13.3].
- **1e:** t ≈ −321.5, so I reject β2 = 1, but 1 isn't a meaningful hypothesis given the units.
- **1f:** Autocorrelation, spurious regression, endogeneity and old data make the result weak. I
  wouldn't recommend tax exemptions based on this study.
- **2a:** N = 1665, R² = 0.008. One more cigarette per day goes with 9.7 g lower birth weight. The
  other variables are small and not significant.
- **2b:** cigs is significant at 1 % (t = −2.76, p = 0.006).
- **2c:** `npvis` (or `monpre`) and `male`.
- **2d:** cigs is significant at 1 %. cigs and motheduc are significant at 5 % and 10 %.
- **2e:** Both intervals contain 0, so neither parent's education is significant at 5 %.

**Choices I made:** The drinking variable is called `drink` in `bwght2`. Rows with missing values in
the model variables are dropped (1832 to 1665). Part 1 uses 72 − 10 = 62 degrees of freedom, and
critical values and p-values come from `scipy.stats.t`. In 1b I used a one-sided test because theory
says which direction the effect should go, and I show the two-sided result too.